# Dataset explorer

Set `dataset_dir` to an existing Parquet dataset directory. Loading returns a Polars LazyFrame; only the preview cell reads rows. Run once in this environment: `pip install -e ".[notebooks,vis]"`. The `vis` extra includes Kaleido for the SVG renderer and static image export.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import plotly.io as pio

from pathlib import Path

from comm_research.infra.paths import PROJECT_ROOT
from comm_research.infra.lake.parquet import scan_parquet
from comm_research.infra.tools.vis import *

In [ ]:
DATA_DIR = PROJECT_ROOT / "data/lake/weather/era5/t_2m"  

In [ ]:
def load_data(data_dir):

    path = Path(data_dir)
    return scan_parquet(path).collect()

def save(df: pl.DataFrame | pl.LazyFrame, filepath: str | Path) -> Path:
    clean_dir = (PROJECT_ROOT / "data" / "clean").resolve()
    relative_path = Path(filepath)

    if relative_path.is_absolute():
        raise ValueError("filepath must be relative to data/clean")

    path = (clean_dir / relative_path).with_suffix(".parquet").resolve()
    
    if not path.is_relative_to(clean_dir):
        raise ValueError("filepath must stay within data/clean")

    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(df, pl.LazyFrame):
        df.sink_parquet(path, compression="zstd")
    else:
        df.write_parquet(path, compression="zstd")
    return path


In [ ]:
df = scan_parquet(DATA_DIR)  # Keep hourly partitions lazy; aggregate after filtering.


In [ ]:
df.limit(5).collect()


### ERA5 weekly actuals matched to GEFS sampling

This section reads hourly Parquet lazily, selects **00, 06, 12 and 18 UTC**, and averages the 28 samples in each Monday-starting `[week_start, week_start + 7 days)` window. It samples instantaneous hourly values; it does not calculate six-hour block means. Temperatures stay in Kelvin.

`ID_DIRECTORY` supplies the existing coordinate-to-ID mapping, so filenames continue to match the GEFS grid IDs. Existing hourly weekly CSVs are preserved. Matched files use the same basenames under `OUTPUT_DIRECTORY`; use that directory when joining matched actuals to GEFS. The mean is `2m_mean`; extrema are explicitly named `2m_max_6h` and `2m_min_6h` and refer only to sampled hours.

Only weeks with exactly **28 distinct, finite samples** are exported. Missing/partial weeks, including boundary weeks, are recorded under `_audit/coverage.csv`; they are not filled or shifted. Duplicate samples, conflicting units or invalid values stop export. Audit files stay in a subdirectory so forecast tools scanning top-level CSVs do not mistake them for actuals.

Set DATA_DIR above to one consistent hourly Parquet dataset (regional `value` schema or global `temperature_2m` schema). Naive timestamps are interpreted as UTC, matching the ERA5 source convention; timezone-aware timestamps are converted to UTC. Optional START/END filter sample timestamps, END exclusive; use Monday 00 UTC boundaries to request only full weeks.


In [ ]:
REGION = "socal"
ID_DIRECTORY = PROJECT_ROOT / "data/clean/weather/era5/t_2m" / REGION
OUTPUT_DIRECTORY = PROJECT_ROOT / "data/clean/weather/era5/t_2m" / f"{REGION}_gefs_6h"
START = None  # Optional datetime.date, inclusive UTC sample date.
END = None    # Optional datetime.date, exclusive UTC sample date.


In [ ]:
import os
import tempfile
from datetime import datetime, time, timezone


def existing_grid_ids(directory, region):
    rows = []
    for path in sorted(Path(directory).glob(f"{region}_temp_2m_*.csv")):
        coordinates = pl.read_csv(path, columns=["lat", "lon"]).unique()
        if coordinates.height != 1 or coordinates.null_count().row(0) != (0, 0):
            raise ValueError(f"Expected one coordinate pair in {path}")
        grid_id = int(path.stem.rsplit("_", 1)[1])
        rows.append({"grid_id": grid_id, **coordinates.row(0, named=True), "filename": path.name})
    if not rows:
        raise ValueError(f"No existing grid-ID CSVs found in {directory}")
    grid = pl.DataFrame(rows).sort("grid_id")
    if grid["grid_id"].n_unique() != grid.height or grid.unique(subset=["lat", "lon"]).height != grid.height:
        raise ValueError("Grid IDs and coordinate pairs must both be unique")
    if grid.filter(~pl.col("lat").is_finite() | ~pl.col("lon").is_finite()
                   | ~pl.col("lat").is_between(-90, 90)
                   | ~pl.col("lon").is_between(-180, 180, closed="left")).height:
        raise ValueError("Invalid coordinates in existing ID files")
    return grid


def matched_era5_weeks(hourly, grid, start=None, end=None):
    if start and end and start >= end:
        raise ValueError("START must precede exclusive END")
    hourly = hourly.lazy() if isinstance(hourly, pl.DataFrame) else hourly
    schema = hourly.collect_schema()
    value_columns = [name for name in ("value", "temperature_2m") if name in schema]
    if len(value_columns) != 1:
        raise ValueError("Select one hourly dataset with value or temperature_2m, not mixed schemas")
    dtype = schema["timestamp"]
    if not isinstance(dtype, pl.Datetime):
        raise ValueError("Hourly Parquet timestamp must have a Datetime dtype")
    stamp = pl.col("timestamp")
    stamp = stamp.dt.convert_time_zone("UTC") if dtype.time_zone else stamp.dt.replace_time_zone("UTC")
    if "variable" in schema:
        hourly = hourly.filter(pl.col("variable") == "2m_temperature")
    hourly = hourly.select(
        stamp.alias("timestamp"), pl.col("latitude").cast(pl.Float64).alias("lat"),
        pl.col("longitude").cast(pl.Float64).alias("lon"),
        pl.col(value_columns[0]).cast(pl.Float64).alias("value"),
        (pl.col("unit") == "K").alias("unit_ok") if "unit" in schema else pl.lit(True).alias("unit_ok"))
    if start:
        hourly = hourly.filter(pl.col("timestamp") >= datetime.combine(start, time(), timezone.utc))
    if end:
        hourly = hourly.filter(pl.col("timestamp") < datetime.combine(end, time(), timezone.utc))
    selected = (hourly.filter(
        pl.col("lat").is_between(grid["lat"].min(), grid["lat"].max())
        & pl.col("lon").is_between(grid["lon"].min(), grid["lon"].max())
        & pl.col("timestamp").dt.hour().is_in([0, 6, 12, 18])
        & (pl.col("timestamp").dt.minute() == 0) & (pl.col("timestamp").dt.second() == 0)
        & (pl.col("timestamp").dt.nanosecond() == 0))
        .join(grid.select("grid_id", "lat", "lon").lazy(), on=["lat", "lon"], how="inner")
        .with_columns(pl.col("timestamp").dt.truncate("1w").alias("week_start")))
    weekly = (selected.group_by("grid_id", "week_start").agg(
        pl.len().alias("n_samples"), pl.col("timestamp").n_unique().alias("n_unique"),
        ((~pl.col("value").is_finite()) | (~pl.col("unit_ok"))).fill_null(True).sum().alias("invalid_samples"),
        pl.col("value").mean().alias("2m_mean"),
        pl.col("value").max().alias("2m_max_6h"), pl.col("value").min().alias("2m_min_6h"))
        .collect(engine="streaming"))
    if weekly.is_empty():
        raise ValueError("No matching 00/06/12/18 UTC ERA5 samples; check DATA_DIR, dates and grid IDs")
    calendar = pl.DataFrame({"week_start": pl.datetime_range(
        weekly["week_start"].min(), weekly["week_start"].max(), interval="1w", eager=True)})
    coverage = (grid.join(calendar, how="cross").join(weekly, on=["grid_id", "week_start"], how="left")
        .with_columns(pl.col("n_samples", "n_unique", "invalid_samples").fill_null(0))
        .with_columns(pl.when(pl.col("invalid_samples") > 0).then(pl.lit("invalid"))
            .when(pl.col("n_samples") != pl.col("n_unique")).then(pl.lit("duplicate"))
            .when(pl.col("n_samples") == 0).then(pl.lit("missing"))
            .when(pl.col("n_samples") == 28).then(pl.lit("complete"))
            .otherwise(pl.lit("incomplete")).alias("status"))
        .rename({"week_start": "timestamp"}).sort(["grid_id", "timestamp"]))
    return coverage


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(dir=path.parent, suffix=".partial")
    os.close(fd)
    try:
        frame.write_csv(temporary)
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def export_matched_actuals(coverage, grid, output, id_directory):
    output = Path(output).resolve()
    if not output.is_relative_to((PROJECT_ROOT / "data/clean").resolve()) or output == Path(id_directory).resolve():
        raise ValueError("Use a separate output directory under data/clean; preserve hourly actuals")
    atomic_csv(coverage, output / "_audit/coverage.csv")
    if coverage.filter(pl.col("status").is_in(["invalid", "duplicate"])).height:
        raise ValueError("Invalid or duplicate source samples; inspect _audit/coverage.csv before exporting")
    complete = coverage.filter(pl.col("status") == "complete")
    if complete.is_empty():
        raise ValueError("No complete 28-sample weeks; inspect _audit/coverage.csv")
    exported = complete.select(
        "grid_id", "timestamp", "lat", "lon", "2m_mean", "2m_max_6h", "2m_min_6h", "n_samples"
    ).with_columns(pl.col("timestamp").dt.offset_by("7d").alias("valid_end"),
                   pl.lit(6).alias("sampling_hours"), pl.lit("K").alias("units"),
                   pl.lit("mean_of_6h_instantaneous_samples").alias("aggregation"))
    for row in grid.iter_rows(named=True):
        atomic_csv(exported.filter(pl.col("grid_id") == row["grid_id"]), output / row["filename"])
    atomic_csv(grid, output / "_audit/grid_ids.csv")
    return exported


In [ ]:
grid_ids = existing_grid_ids(ID_DIRECTORY, REGION)
coverage = matched_era5_weeks(df, grid_ids, start=START, end=END)
matched_actuals = export_matched_actuals(coverage, grid_ids, OUTPUT_DIRECTORY, ID_DIRECTORY)
print(f"Wrote {grid_ids.height} grid CSVs and {matched_actuals.height:,} complete grid-weeks to {OUTPUT_DIRECTORY}")
coverage.group_by("status").len().sort("status")
